In [1]:
# ============================================================
# 셀 1 - imports
# ============================================================
import requests
import json
import re
import numpy as np
import pandas as pd
import folium
from folium.plugins import HeatMap
from pyproj import Transformer
from pathlib import Path
from pydantic import BaseModel
from typing import Optional
from sklearn.neighbors import BallTree

transformer = Transformer.from_crs("EPSG:5179", "EPSG:4326", always_xy=True)

In [2]:
# ============================================================
# 셀 2 - Pydantic 모델
# ============================================================
class AccidentInfo(BaseModel):
    acdnt_no: str
    serial_no: str
    acdnt_year: str
    acdnt_dd_dc: str
    dfk_dc: str
    tmzon_div_1_dc: str
    occrrnc_time_dc: str
    x_crdnt: float
    y_crdnt: float
    legaldong_code: str
    legaldong_name: str
    acdnt_hdc: str
    acdnt_dc: str
    road_stle_dc: str
    acdnt_gae_dc: str
    acdnt_age_2_dc: Optional[str] = None
    dmge_vhcle_asort_dc: Optional[str] = None
    injury_dgree_2_dc: Optional[str] = None
    wrngdo_vhcle_asort_dc: Optional[str] = None
    lrg_violt_1_dc: Optional[str] = None
    wether_sttus_dc: Optional[str] = None
    rdse_sttus_dc: Optional[str] = None
    dprs_cnt: int = 0
    sep_cnt: int = 0
    slp_cnt: int = 0
    lat: Optional[float] = None
    lon: Optional[float] = None

class ResultValue(BaseModel):
    accidentInfoList: list[AccidentInfo]

class TaasResponse(BaseModel):
    resultValue: ResultValue

In [3]:
# ============================================================
# 셀 3 - 크롤링 함수
# ============================================================
headers_base = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
}

def get_session_and_csrf() -> tuple[requests.Session, str]:
    session = requests.Session()
    response = session.get(
        "https://taas.koroad.or.kr/gis/mcm/mcl/initMap.do?menuId=GIS_GMP_STS_RSN",
        headers=headers_base
    )
    match = re.search(r'<meta name="_csrf"[^>]+content="([^"]+)"', response.text)
    token = match.group(1)
    print(f"CSRF: {token}")
    return session, token

def crawl_and_save() -> pd.DataFrame:
    session, csrf_token = get_session_and_csrf()
    headers = {
        **headers_base,
        "Accept": "application/json, text/javascript, */*; q=0.01",
        "Content-Type": "application/json;charset=UTF-8",
        "X-CSRF-TOKEN": csrf_token,
        "X-Requested-With": "XMLHttpRequest",
        "Referer": "https://taas.koroad.or.kr/gis/mcm/mcl/initMap.do?menuId=GIS_GMP_STS_RSN",
        "Origin": "https://taas.koroad.or.kr",
    }
    payload = {
        "searchType": "00",
        "pageIndex": 1,
        "zoneYn": False,
        "engnCode": "00",
        "startAcdntYear": "2024",
        "endAcdntYear": "2024",
        "legaldongCode": "11%",
        "acdntGaeCode": "01,02,03,04",
        "searchSimpleCondition": "42"
    }
    response = session.post(
        "https://taas.koroad.or.kr/gis/srh/ash/selectAccidentInfo.do",
        headers=headers, json=payload
    )
    response.raise_for_status()
    data = response.json()
    parsed = TaasResponse(**data)
    accidents = parsed.resultValue.accidentInfoList
    for acc in accidents:
        lon, lat = transformer.transform(acc.x_crdnt, acc.y_crdnt)
        acc.lat = lat
        acc.lon = lon
    Path("data/raw").mkdir(parents=True, exist_ok=True)
    Path("data/processed").mkdir(parents=True, exist_ok=True)
    with open("data/raw/taas_seoul_elderly_pedestrian_2024.json", "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)
    df = pd.DataFrame([acc.model_dump() for acc in accidents])
    df.to_csv("data/processed/taas_seoul_elderly_pedestrian_2024.csv", index=False, encoding="utf-8-sig")
    print(f"총 {len(df)}건 저장완료")
    return df

In [4]:
# ============================================================
# 셀 4 - 크롤링 실행 (이미 했으면 스킵)
# ============================================================
# accidents_df = crawl_and_save()  # 이미 했으면 주석 처리
accidents_df = pd.read_csv("data/processed/taas_seoul_elderly_pedestrian_2024.csv")
dongjakgu = accidents_df[accidents_df['legaldong_name'].str.contains('동작구')]
print(f"동작구 노인 보행사고: {len(dongjakgu)}건")

동작구 노인 보행사고: 82건


In [5]:
# ============================================================
# 셀 5 - 횡단보도 데이터 로드 및 위험 구간 계산
# ============================================================
df = pd.read_csv("서울특별시_동작구_횡단보도_20260306.csv")

# 전체 횡단보도
all_crosswalk = df.copy()

# 신호시간 있는 것
signal_df = df[df['보행자신호등유무'] == 'Y'].copy()
signal_df['녹색신호시간'] = pd.to_numeric(signal_df['녹색신호시간'], errors='coerce')
signal_df = signal_df.dropna(subset=['녹색신호시간'])
signal_df['필요시간'] = signal_df['횡단보도연장'] / 0.8
signal_df['부족시간'] = signal_df['필요시간'] - signal_df['녹색신호시간']
signal_df['위험여부'] = signal_df['부족시간'] > 0

# 위험 횡단보도
danger = signal_df[signal_df['위험여부']].copy().reset_index(drop=True)

# BallTree 매칭
cw_coords = np.radians(danger[['위도', '경도']].values)
acc_coords = np.radians(dongjakgu[['lat', 'lon']].dropna().values)
tree = BallTree(cw_coords, metric='haversine')
indices_200 = tree.query_radius(acc_coords, r=200/6371000)
danger['주변사고수_200m'] = 0
for idx_list in indices_200:
    for idx in idx_list:
        danger.loc[idx, '주변사고수_200m'] += 1

print(f"전체 횡단보도: {len(all_crosswalk)}개")
print(f"신호시간 있는 것: {len(signal_df)}개")
print(f"위험 횡단보도: {len(danger)}개")

전체 횡단보도: 1057개
신호시간 있는 것: 310개
위험 횡단보도: 21개


In [6]:
# ============================================================
# 셀 5-1 - 도로형태 분석 + 무신호 횡단보도 위험 매칭
# ============================================================

# --- 도로형태별 사고 분포 ---
print("=== 도로형태별 노인 보행사고 (동작구) ===")
road_counts = dongjakgu['road_stle_dc'].value_counts()
for road, cnt in road_counts.items():
    pct = cnt / len(dongjakgu) * 100
    print(f"  {road}: {cnt}건 ({pct:.1f}%)")

# --- 단일로 사고 ---
single_road = dongjakgu[dongjakgu['road_stle_dc'] == '단일로 - 기타']
print(f"\n단일로(골목길/이면도로) 사고: {len(single_road)}건 / {len(dongjakgu)}건 ({len(single_road)/len(dongjakgu)*100:.1f}%)")

# --- 무신호 횡단보도 ---
no_signal = df[df['보행자신호등유무'] == 'N'].copy()
print(f"무신호 횡단보도: {len(no_signal)}개 / {len(df)}개 ({len(no_signal)/len(df)*100:.1f}%)")

# --- 무신호 횡단보도 주변 사고 매칭 (BallTree) ---
acc_all_coords = np.radians(dongjakgu[['lat', 'lon']].dropna().values)
ns_coords = np.radians(no_signal[['위도', '경도']].dropna().values)

tree_acc = BallTree(acc_all_coords, metric='haversine')
acc_near_ns = tree_acc.query_radius(ns_coords, r=200/6371000, count_only=True)
no_signal['주변사고수_200m'] = acc_near_ns

# --- 무신호 위험 횡단보도 (주변 200m 내 사고 2건 이상) ---
no_signal_danger = no_signal[no_signal['주변사고수_200m'] >= 2].copy().sort_values('주변사고수_200m', ascending=False).reset_index(drop=True)
print(f"\n무신호 위험 횡단보도 (200m내 사고 2건+): {len(no_signal_danger)}개")
print(no_signal_danger[['소재지도로명주소', '횡단보도연장', '주변사고수_200m']].head(10).to_string())

# --- 단일로 사고 중 무신호 횡단보도 100m 이내 비율 ---
single_coords = np.radians(single_road[['lat', 'lon']].dropna().values)
tree_ns = BallTree(ns_coords, metric='haversine')
ns_near_single = tree_ns.query_radius(single_coords, r=100/6371000, count_only=True)
match_count = sum(ns_near_single > 0)
print(f"\n단일로 사고 중 100m내 무신호 횡단보도 존재: {match_count}/{len(single_road)}건 ({match_count/len(single_road)*100:.1f}%)")

=== 도로형태별 노인 보행사고 (동작구) ===
  단일로 - 기타: 41건 (50.0%)
  교차로 - 교차로횡단보도내: 17건 (20.7%)
  교차로 - 교차로안: 11건 (13.4%)
  교차로 - 교차로부근: 7건 (8.5%)
  기타 - 기타: 6건 (7.3%)

단일로(골목길/이면도로) 사고: 41건 / 82건 (50.0%)
무신호 횡단보도: 733개 / 1057개 (69.3%)

무신호 위험 횡단보도 (200m내 사고 2건+): 145개
               소재지도로명주소  횡단보도연장  주변사고수_200m
0     서울특별시 동작구 성대로1길 2    10.3          11
1     서울특별시 동작구 성대로2길 1     8.7          11
2     서울특별시 동작구 상도로 102     8.3          11
3      서울특별시 동작구 성대로 32     8.5          10
4      서울특별시 동작구 상도로 88    17.4           9
5      서울특별시 동작구 상도로 91     6.5           9
6    서울특별시 동작구 동작대로 129    12.0           8
7  서울특별시 동작구 동작대로 111-2    10.8           8
8  서울특별시 동작구 동작대로 111-2     3.6           8
9  서울특별시 동작구 동작대로 121-4     8.0           7

단일로 사고 중 100m내 무신호 횡단보도 존재: 29/41건 (70.7%)


In [7]:
# ============================================================
# 셀 6 - 카카오맵 HTML 생성 (서울 전체 구별/차량종류 필터 + 동작구 횡단보도 분석)
# ============================================================
kakao_key = "4827d1df867dfc08ae1daba2b1d25835"

# --- 서울 전체 사고 데이터 (구 이름 추출) ---
accidents_all = accidents_df.copy()
accidents_all['구'] = accidents_all['legaldong_name'].str.extract(r'서울특별시 (\S+구)')[0]
gu_list = sorted(accidents_all['구'].dropna().unique().tolist())
gu_counts = accidents_all['구'].value_counts().to_dict()

# 가해차량 종류 목록
vhcle_list = accidents_all['wrngdo_vhcle_asort_dc'].dropna().value_counts()
vhcle_names = vhcle_list.index.tolist()

# 서울 전체 사고 [lat, lon, 구, 법정동, 날짜, 심각도, 도로형태, 가해차량]
all_accident_data = accidents_all[['lat', 'lon', '구', 'legaldong_name', 'acdnt_dd_dc', 'acdnt_gae_dc', 'road_stle_dc', 'wrngdo_vhcle_asort_dc']].dropna(subset=['lat','lon']).values.tolist()

# --- 동작구 횡단보도 데이터 (기존) ---
all_cw_data = all_crosswalk[['위도', '경도', '소재지도로명주소', '횡단보도연장', '보행자신호등유무']].dropna(subset=['위도','경도']).values.tolist()
danger_data = danger[['위도', '경도', '부족시간', '소재지도로명주소', '횡단보도연장', '녹색신호시간', '주변사고수_200m']].values.tolist()
no_signal_danger_data = no_signal_danger[['위도', '경도', '소재지도로명주소', '횡단보도연장', '주변사고수_200m']].values.tolist()

# 구별 옵션 HTML
gu_options = '\n'.join([f'        <option value="{g}">{g} ({gu_counts.get(g, 0)}건)</option>' for g in gu_list])

# 차량종류 옵션 HTML
vhcle_options = '\n'.join([f'        <option value="{v}">{v} ({c}건)</option>' for v, c in vhcle_list.items()])

html_content = f"""<!DOCTYPE html>
<html>
<head>
    <meta charset="utf-8">
    <title>서울시 노인 보행 취약 구간 종합 지도</title>
    <style>
        * {{ margin: 0; padding: 0; box-sizing: border-box; }}
        body {{ font-family: 'Apple SD Gothic Neo', sans-serif; }}
        #map {{ width: 100vw; height: 100vh; }}
        #control {{
            position: absolute; top: 20px; left: 20px;
            background: white; padding: 16px;
            border-radius: 12px; box-shadow: 0 2px 10px rgba(0,0,0,0.2);
            font-size: 13px; z-index: 10; min-width: 260px;
            max-height: calc(100vh - 40px); overflow-y: auto;
        }}
        #control h4 {{
            font-size: 14px; font-weight: bold;
            margin-bottom: 12px; color: #333;
            border-bottom: 2px solid #e74c3c;
            padding-bottom: 6px;
        }}
        .layer-group {{
            margin-bottom: 10px;
            padding-bottom: 8px;
            border-bottom: 1px solid #eee;
        }}
        .layer-group:last-child {{ border-bottom: none; margin-bottom: 0; padding-bottom: 0; }}
        .group-label {{
            font-size: 11px; color: #999; font-weight: bold;
            margin-bottom: 4px;
        }}
        .layer-item {{
            display: flex; align-items: center;
            margin-bottom: 6px; cursor: pointer;
        }}
        .layer-item input[type="checkbox"] {{ margin-right: 8px; cursor: pointer; }}
        .dot {{
            display: inline-block; width: 12px; height: 12px;
            border-radius: 50%; margin-right: 6px; flex-shrink: 0;
        }}
        .filter-select {{
            width: 100%; padding: 6px 8px; border: 1px solid #ddd;
            border-radius: 6px; font-size: 13px; margin-top: 4px;
            cursor: pointer;
        }}
        #info {{
            position: absolute; bottom: 20px; left: 20px;
            background: white; padding: 12px 16px;
            border-radius: 8px; box-shadow: 0 2px 8px rgba(0,0,0,0.15);
            font-size: 12px; z-index: 10; color: #666;
        }}
    </style>
    <script src="//dapi.kakao.com/v2/maps/sdk.js?appkey={kakao_key}&libraries=services,clusterer,drawing"></script>
</head>
<body>
<div id="map"></div>

<div id="control">
    <h4>서울시 노인 보행 취약 구간 종합 지도</h4>

    <div class="layer-group">
        <div class="group-label">구 선택</div>
        <select id="guFilter" class="filter-select">
            <option value="all">서울 전체 ({len(all_accident_data)}건)</option>
{gu_options}
        </select>
    </div>

    <div class="layer-group">
        <div class="group-label">가해차량 종류</div>
        <select id="vhcleFilter" class="filter-select">
            <option value="all">전체 차량</option>
{vhcle_options}
        </select>
    </div>

    <div class="layer-group">
        <div class="group-label">사고 데이터</div>
        <div class="layer-item">
            <input type="checkbox" id="toggleAccident" checked>
            <span class="dot" style="background:#f39c12;"></span>
            <span>노인 보행사고 (<span id="accidentCount">{len(all_accident_data)}</span>건)</span>
        </div>
        <div class="layer-item">
            <input type="checkbox" id="toggleHeat" checked>
            <span class="dot" style="background:rgba(255,100,0,0.6); border-radius:2px;"></span>사고 히트맵
        </div>
    </div>

    <div class="layer-group">
        <div class="group-label">동작구 횡단보도 분석</div>
        <div class="layer-item">
            <input type="checkbox" id="toggleDanger" checked>
            <span class="dot" style="background:#e74c3c;"></span>신호 부족 ({len(danger_data)}개)
        </div>
        <div class="layer-item">
            <input type="checkbox" id="toggleNoSignal" checked>
            <span class="dot" style="background:#9b59b6;"></span>무신호 사고다발 ({len(no_signal_danger_data)}개)
        </div>
        <div class="layer-item">
            <input type="checkbox" id="toggleAllCW">
            <span class="dot" style="background:#3498db;"></span>전체 횡단보도 ({len(all_cw_data)}개)
        </div>
    </div>
</div>

<div id="info">
    <span id="infoText">서울 전체 노인 보행사고 {len(all_accident_data)}건 | 동작구 횡단보도 분석 포함</span>
</div>

<script>
    var map = new kakao.maps.Map(document.getElementById('map'), {{
        center: new kakao.maps.LatLng(37.5665, 126.9780),
        level: 8
    }});

    var allCWMarkers = [];
    var dangerMarkers = [];
    var noSignalMarkers = [];
    var accidentOverlays = [];
    var heatCircles = [];

    var currentGu = 'all';
    var currentVhcle = 'all';
    var accidentVisible = true;
    var heatVisible = true;

    function createDot(style) {{
        var el = document.createElement('div');
        Object.assign(el.style, style);
        return el;
    }}

    // ── 서울 전체 사고 마커 (주황 점)
    var accidentData = {json.dumps(all_accident_data)};
    accidentData.forEach(function(d) {{
        var lat = d[0], lon = d[1];
        var gu = d[2], dong = d[3], date = d[4], severity = d[5], road = d[6], vhcle = d[7] || '불명';
        var el = createDot({{
            width: '10px', height: '10px', background: '#f39c12',
            borderRadius: '50%', border: '1px solid white', cursor: 'pointer',
            boxShadow: '0 1px 4px rgba(0,0,0,0.3)'
        }});
        var overlay = new kakao.maps.CustomOverlay({{
            map: map,
            position: new kakao.maps.LatLng(lat, lon),
            content: el
        }});
        var iw = new kakao.maps.InfoWindow({{
            content: '<div style="padding:10px;font-size:12px;min-width:180px;">' +
                '<b>노인 보행사고</b><br>' +
                '위치: ' + dong + '<br>' +
                '발생: ' + date + '<br>' +
                '심각도: ' + severity + '<br>' +
                '도로형태: ' + road + '<br>' +
                '가해차량: <b>' + vhcle + '</b>' +
                '</div>',
            removable: true
        }});
        el.addEventListener('click', function() {{
            iw.open(map, new kakao.maps.Marker({{ position: new kakao.maps.LatLng(lat, lon) }}));
        }});
        accidentOverlays.push({{ overlay: overlay, gu: gu, vhcle: vhcle }});
    }});

    // ── 히트맵 (원형 오버레이)
    accidentData.forEach(function(d) {{
        var circle = new kakao.maps.Circle({{
            map: map,
            center: new kakao.maps.LatLng(d[0], d[1]),
            radius: 100,
            strokeWeight: 0,
            fillColor: '#ff6600',
            fillOpacity: 0.12
        }});
        heatCircles.push({{ circle: circle, gu: d[2], vhcle: d[7] || '불명' }});
    }});

    // ── 동작구 전체 횡단보도 (파란 점, 기본 OFF)
    var allCWData = {json.dumps(all_cw_data)};
    allCWData.forEach(function(d) {{
        var el = createDot({{
            width: '8px', height: '8px', background: '#3498db',
            borderRadius: '50%', opacity: '0.7', border: '1px solid white'
        }});
        var overlay = new kakao.maps.CustomOverlay({{
            position: new kakao.maps.LatLng(d[0], d[1]),
            content: el
        }});
        var iw = new kakao.maps.InfoWindow({{
            content: '<div style="padding:10px;font-size:12px;min-width:180px;">' +
                '<b>' + d[2] + '</b><br>' +
                '횡단보도 길이: ' + d[3] + 'm<br>' +
                '보행자신호등: ' + d[4] +
                '</div>',
            removable: true
        }});
        el.addEventListener('click', function() {{
            iw.open(map, new kakao.maps.Marker({{ position: new kakao.maps.LatLng(d[0], d[1]) }}));
        }});
        allCWMarkers.push(overlay);
    }});

    // ── 신호 부족 횡단보도 (빨간 점)
    var dangerData = {json.dumps(danger_data)};
    dangerData.forEach(function(d) {{
        var lat = d[0], lon = d[1];
        var shortage = d[2], addr = d[3], length = d[4], green = d[5], accidents = d[6];
        var size = (shortage > 10 ? 18 : shortage > 5 ? 14 : 10) + 'px';
        var el = createDot({{
            width: size, height: size, background: '#e74c3c',
            borderRadius: '50%', border: '2px solid white', cursor: 'pointer',
            boxShadow: '0 2px 6px rgba(0,0,0,0.4)'
        }});
        var overlay = new kakao.maps.CustomOverlay({{
            map: map,
            position: new kakao.maps.LatLng(lat, lon),
            content: el
        }});
        var iw = new kakao.maps.InfoWindow({{
            content: '<div style="padding:12px;font-size:12px;min-width:220px;">' +
                '<b style="font-size:13px;">' + addr + '</b><br><br>' +
                '<span style="color:#e74c3c;font-weight:bold;">신호 부족 횡단보도</span><br>' +
                '횡단보도 길이: <b>' + length + 'm</b><br>' +
                '녹색신호: <b>' + green + '초</b><br>' +
                '필요시간: <b>' + (length/0.8).toFixed(1) + '초</b><br>' +
                '부족시간: <b style="color:red;">' + shortage.toFixed(1) + '초</b><br><br>' +
                '주변 노인사고(200m): <b style="color:#e74c3c;">' + accidents + '건</b>' +
                '</div>',
            removable: true
        }});
        el.addEventListener('click', function() {{
            iw.open(map, new kakao.maps.Marker({{ position: new kakao.maps.LatLng(lat, lon) }}));
        }});
        dangerMarkers.push(overlay);
    }});

    // ── 무신호 사고다발 횡단보도 (보라 점)
    var noSignalData = {json.dumps(no_signal_danger_data)};
    noSignalData.forEach(function(d) {{
        var lat = d[0], lon = d[1];
        var addr = d[2], length = d[3], accidents = d[4];
        var size = (accidents >= 4 ? 16 : accidents >= 3 ? 13 : 10) + 'px';
        var el = createDot({{
            width: size, height: size, background: '#9b59b6',
            borderRadius: '50%', border: '2px solid white', cursor: 'pointer',
            boxShadow: '0 2px 6px rgba(0,0,0,0.4)'
        }});
        var overlay = new kakao.maps.CustomOverlay({{
            map: map,
            position: new kakao.maps.LatLng(lat, lon),
            content: el
        }});
        var iw = new kakao.maps.InfoWindow({{
            content: '<div style="padding:12px;font-size:12px;min-width:220px;">' +
                '<b style="font-size:13px;">' + addr + '</b><br><br>' +
                '<span style="color:#9b59b6;font-weight:bold;">무신호 사고다발 횡단보도</span><br>' +
                '횡단보도 길이: <b>' + length + 'm</b><br>' +
                '보행자신호등: <b style="color:#9b59b6;">없음</b><br><br>' +
                '주변 노인사고(200m): <b style="color:#e74c3c;">' + accidents + '건</b>' +
                '</div>',
            removable: true
        }});
        el.addEventListener('click', function() {{
            iw.open(map, new kakao.maps.Marker({{ position: new kakao.maps.LatLng(lat, lon) }}));
        }});
        noSignalMarkers.push(overlay);
    }});

    // ── 필터링 함수 (구 + 차량종류 동시 적용)
    function applyFilters() {{
        var visibleCount = 0;

        accidentOverlays.forEach(function(item) {{
            var guMatch = currentGu === 'all' || item.gu === currentGu;
            var vhcleMatch = currentVhcle === 'all' || item.vhcle === currentVhcle;
            var show = accidentVisible && guMatch && vhcleMatch;
            item.overlay.setMap(show ? map : null);
            if (show) visibleCount++;
        }});

        heatCircles.forEach(function(item) {{
            var guMatch = currentGu === 'all' || item.gu === currentGu;
            var vhcleMatch = currentVhcle === 'all' || item.vhcle === currentVhcle;
            var show = heatVisible && guMatch && vhcleMatch;
            item.circle.setMap(show ? map : null);
        }});

        document.getElementById('accidentCount').textContent = visibleCount;

        var guLabel = currentGu === 'all' ? '서울 전체' : currentGu;
        var vhcleLabel = currentVhcle === 'all' ? '' : ' | ' + currentVhcle;
        document.getElementById('infoText').textContent =
            guLabel + ' 노인 보행사고 ' + visibleCount + '건' + vhcleLabel;
    }}

    // ── 레이어 토글
    function toggleLayer(markers, show) {{
        markers.forEach(function(m) {{ m.setMap(show ? map : null); }});
    }}

    document.getElementById('guFilter').addEventListener('change', function() {{
        currentGu = this.value;
        applyFilters();
    }});
    document.getElementById('vhcleFilter').addEventListener('change', function() {{
        currentVhcle = this.value;
        applyFilters();
    }});
    document.getElementById('toggleAccident').addEventListener('change', function() {{
        accidentVisible = this.checked;
        applyFilters();
    }});
    document.getElementById('toggleHeat').addEventListener('change', function() {{
        heatVisible = this.checked;
        applyFilters();
    }});
    document.getElementById('toggleDanger').addEventListener('change', function() {{
        toggleLayer(dangerMarkers, this.checked);
    }});
    document.getElementById('toggleNoSignal').addEventListener('change', function() {{
        toggleLayer(noSignalMarkers, this.checked);
    }});
    document.getElementById('toggleAllCW').addEventListener('change', function() {{
        toggleLayer(allCWMarkers, this.checked);
    }});
</script>
</body>
</html>"""

with open("kakao_map.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print(f"저장 완료: kakao_map.html")
print(f"서울 전체 사고: {len(all_accident_data)}건, {len(gu_list)}개 구, {len(vhcle_names)}개 차량종류")

저장 완료: kakao_map.html
서울 전체 사고: 1963건, 25개 구, 10개 차량종류
